In [1]:
import os

In [2]:
with open('data/fake.txt', 'r', encoding = 'utf-8', errors = 'ignore') as file:
    text = file.read()

In [3]:
print(len(text))
print(text[:500])


62700077
title,text,subject,date
 Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing,"Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very di


In [4]:
def chunck_text(text, max_chars = 500):
    chunks = []
    text_as_list = text.strip().split('\n') 
    for line in text_as_list:
        while len(line) > max_chars:
            chunk = line[:max_chars]
            line = line[max_chars:]
            chunks.append(chunk)
        if line:
            chunks.append(line)  
    return chunks     

In [41]:
def chunk_text2(text, max_chars=600):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    
    chunks = []
    current = ""

    for p in paragraphs:
        if len(current) + len(p) + 1 <= max_chars:
            current = (current + " " + p).strip()
        else:
            if current:
                chunks.append(current)
            current = p

    if current:
        chunks.append(current)

    return chunks


In [42]:
chunks = chunk_text2(text)

In [43]:
len(chunks)

23068

In [44]:
print(chunks[1][:40])

Donald Trump Sends Out Embarrassing New 


In [45]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [162]:
cosine_similarity([[1,1,1]],[[3,3,3],[3,3,3],[3,3,3],[3,3,3]])

array([[1., 1., 1., 1.]])

In [47]:
# we will load an embedding model
print('\nLoading Embedding Model ...')
model = SentenceTransformer('all-miniLM-L6-v2')
print('\nDone!')

# embed all chunks
print('\nEncoding chunks...')
chunk_embeddings = model.encode(chunks, convert_to_numpy= True, normalize_embeddings= True)

print(f'Embeddings shape :{chunk_embeddings.shape}')


Loading Embedding Model ...

Done!

Encoding chunks...
Embeddings shape :(23068, 384)


In [160]:
def research(query, model, chunks,  chunk_embedding, k = 3):
    # Embed query
    query_embedding = model.encode(query, convert_to_numpy = True, normalize_embeddings = True)
    # calculate similarity
    print(len(chunk_embedding[0]))
    print(query_embedding[0])
    print(len(query_embedding))

    
    sim = cosine_similarity([query_embedding], chunk_embedding)
    # find similar chunks
    print(len(sim[0]))
    top_idx = sim[0].argsort()[::-1][:k]
    print(len(top_idx))
    print(top_idx)


    print('\n' + '='*60)
    print(f'Query: {query}')
    print('\n' + '~'*60)
    for rank, idx in enumerate(top_idx, start= 1):
        print(idx)
        print(f'{rank})- similarity= {sim[0][idx]} | text: {chunks[idx]}')
        print('='*40)
    return top_idx


In [161]:
query = 'Trump saying bad things against immigrants'
research(query, model= model, chunks= chunks, chunk_embedding= chunk_embeddings, k=3)

384
0.088387065
384
23068
3
[4121 8066 4753]

Query: Trump saying bad things against immigrants

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
4121
1)- similarity= 0.6444690227508545 | text: Trump Just Said The Most Racist Thing Imaginable About Latino Immigrants,"The only way that Trump can talk about immigration, and immigrants, is in racist terms. For someone other than, well, someone in a position of power, it d probably be easy to explain his use of the term,  bad hombres,  as simple ignorance. Maybe. With him and his racist tendencies, though,it s not, and he proved that during the final presidential debate.In talking about undocumented immigrants and the crime they re supposedly committing, he said: But we have some bad hombres here that were going to get them out. Because that s what everyone wants, supposedly. What he misses isn t just that his border wall and deportation force plans aren t realistic   he also misses the fact that saying,  bad hombres,  is horri

array([4121, 8066, 4753])

In [157]:
chunks[4121]

'Trump Just Said The Most Racist Thing Imaginable About Latino Immigrants,"The only way that Trump can talk about immigration, and immigrants, is in racist terms. For someone other than, well, someone in a position of power, it d probably be easy to explain his use of the term,  bad hombres,  as simple ignorance. Maybe. With him and his racist tendencies, though,it s not, and he proved that during the final presidential debate.In talking about undocumented immigrants and the crime they re supposedly committing, he said: But we have some bad hombres here that were going to get them out. Because that s what everyone wants, supposedly. What he misses isn t just that his border wall and deportation force plans aren t realistic   he also misses the fact that saying,  bad hombres,  is horribly racist. Why is that?Using a culture s own language to attack them is inherently racist.  Hombre  means  man,  in Spanish, and specifically refers to  tough men.  But using it against them like that is 

In [248]:
q= 'test'
ql = ['embed', 7, 'thing']
q_emb = model.encode(q)
q_emb_np = model.encode(q, convert_to_numpy= True)

In [249]:
print(q_emb)

[ 1.15734367e-02  2.51362026e-02 -3.67018469e-02  5.93248829e-02
 -7.14904908e-03 -4.11942229e-02  7.70873949e-02  3.74425687e-02
  1.24489907e-02 -6.11762889e-03  1.70342755e-02 -7.70153850e-02
 -3.94164264e-04  2.79090628e-02 -1.59891583e-02 -6.82752728e-02
  8.88464600e-03 -2.02807039e-02 -8.03599581e-02 -1.30740469e-02
 -4.11000140e-02 -2.58980803e-02 -2.65386701e-02  3.30522880e-02
 -2.20791958e-02  2.10461039e-02 -5.79220019e-02  3.29487622e-02
  2.97073852e-02 -6.22484051e-02  3.87880243e-02  3.19906883e-02
  1.53308157e-02  4.53069732e-02  5.31494431e-02  1.33606764e-02
  4.12249267e-02  2.81429105e-02  1.93984341e-02 -3.25232325e-03
 -3.61234089e-03 -1.42860249e-01  3.80711704e-02 -1.09162051e-02
  2.60940064e-02  4.13699485e-02 -1.60157550e-02  5.35600968e-02
 -5.68594411e-02  1.22468015e-02 -3.49966064e-02 -3.97541821e-02
 -4.61429916e-02 -3.91123667e-02 -1.80036407e-02  2.16342863e-02
 -6.46134187e-03 -2.65694913e-02  4.87288013e-02  4.34006080e-02
  4.62000854e-02 -3.44790

In [104]:
len(q_emb_np)

384

In [191]:
def search(query, model, chunks,  chunk_embedding, k = 3):
    # Embed query
    query_embedding = model.encode(query, convert_to_numpy = True, normalize_embeddings = True)
    # calculate similarity    
    sim = cosine_similarity([query_embedding], chunk_embedding)
    # find similar chunks
    top_idx = sim[0].argsort()[::-1][:k]
    top_chunks = []
    for rank, idx in enumerate(top_idx, start= 1):
        top_chunks.append({'similarity' : sim[0][idx], 'text': chunks[idx]})
    chunks_only = []
    for chunk in top_chunks:
        text = chunk['text']
        chunks_only.append(text)
    return top_chunks, chunks_only


In [234]:
query = 'What didi Trum say about immigrants ?'
retrieved_chunks_sims, retrieved_chunks =search(query, model= model, chunks= chunks, chunk_embedding= chunk_embeddings, k=3)

In [235]:
retrieved_chunks_sims

[{'similarity': np.float32(0.4747157),
  'text': 'Ben Carson Just Called Slaves ‘Immigrants’ As Part Of Anti-Immigrant Speech (VIDEO),"Trump s newly confirmed Secretary of Housing and Urban Development just dropped a doozy on us. No, not the one about Harriet Tubman. This is a new WTF moment for the sleepy-eyed Ben Carson. In a talk with department employees, Carson referred to slaves as  immigrants  dreaming of a better life, The Hill reports. Sure, except slaves didn t want to come to this country. They had families, homes, a life and a country. Slaves were shackled and brought to the U.S. in deplorable circumstances, beaten within an inch of their lives. That s what America is about. A land of dreams and opportunity,  Carson said, while inadvertently touting the very reason we should allow refugees to enter our borders.  There were other immigrants who came here in the bottom of slave ships, worked even longer, even harder for less. Ben Carson at HUD:  There were other immigrants wh

In [236]:
retrieved_chunks

['Ben Carson Just Called Slaves ‘Immigrants’ As Part Of Anti-Immigrant Speech (VIDEO),"Trump s newly confirmed Secretary of Housing and Urban Development just dropped a doozy on us. No, not the one about Harriet Tubman. This is a new WTF moment for the sleepy-eyed Ben Carson. In a talk with department employees, Carson referred to slaves as  immigrants  dreaming of a better life, The Hill reports. Sure, except slaves didn t want to come to this country. They had families, homes, a life and a country. Slaves were shackled and brought to the U.S. in deplorable circumstances, beaten within an inch of their lives. That s what America is about. A land of dreams and opportunity,  Carson said, while inadvertently touting the very reason we should allow refugees to enter our borders.  There were other immigrants who came here in the bottom of slave ships, worked even longer, even harder for less. Ben Carson at HUD:  There were other immigrants who came here in the bottom of slave ships, worked

In [237]:
print(f"{retrieved_chunks_sims[0]['similarity']} + {retrieved_chunks_sims[1]['similarity']} = {retrieved_chunks_sims[0]['similarity'] + retrieved_chunks_sims[1]['similarity']}")

0.4747157096862793 + 0.4624284505844116 = 0.9371441602706909


In [180]:
import requests

In [213]:
def testLLMresponse(model_name, test_message):
    url = 'http://localhost:11434/api/generate'
    data = {
        'model': model_name,
        'prompt': test_message,
        'stream': False
    }
    resp = requests.post(url = url, json= data)
    return resp.json()

In [222]:
def promptLLM(system_prompt, user_prompt, model_name):
    
    url = 'http://localhost:11434/api/generate'
    
    data = {
        'model': model_name,
        'prompt': f" {system_prompt}.\nAre you ready ? Here is the context:\n{user_prompt} \n\nQuestion: {query}\n\nRemember, answer based ONLY on the context above.",
        'stream': False    
    } 


    """ 
        data = {
        'model': model_name,
        'messages':[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
            ]        
    } 
    """

    resp = requests.post(url = url, json= data, stream= False)
    resp.raise_for_status()
    out = resp.json()
    return(out['response'])


In [244]:
system_prompt=( "You are an assistant that answers questions based ONLY on the provided context. "
        "Give clearest full answer based on the context only, and quote from context to back up your answer"
        "BUT, If the context does not contain the answer, say it is not mentionned. "
        "Do not invent information."
        "At the end, make a separation from previous text, and answer from your own knowledge even out of provided context. \n\n")
        
context_text = "\n\n".join(retrieved_chunks)
user_prompt= f"Context:\n{context_text}\n\nQuestion: What did Trump say about immigrants ?\n\nAnswer based only on the context above."
model_name = 'llama3'

In [239]:
context_text

'Ben Carson Just Called Slaves ‘Immigrants’ As Part Of Anti-Immigrant Speech (VIDEO),"Trump s newly confirmed Secretary of Housing and Urban Development just dropped a doozy on us. No, not the one about Harriet Tubman. This is a new WTF moment for the sleepy-eyed Ben Carson. In a talk with department employees, Carson referred to slaves as  immigrants  dreaming of a better life, The Hill reports. Sure, except slaves didn t want to come to this country. They had families, homes, a life and a country. Slaves were shackled and brought to the U.S. in deplorable circumstances, beaten within an inch of their lives. That s what America is about. A land of dreams and opportunity,  Carson said, while inadvertently touting the very reason we should allow refugees to enter our borders.  There were other immigrants who came here in the bottom of slave ships, worked even longer, even harder for less. Ben Carson at HUD:  There were other immigrants who came here in the bottom of slave ships, worked 

In [230]:
import textwrap

In [245]:
answer = promptLLM(system_prompt, user_prompt, model_name)
print(textwrap.fill(answer, width= 80))

According to the provided context:  **Trump's statement about immigrants**  From
the first article:  "There were other immigrants who came here in the bottom of
slave ships, worked even longer, even harder for less. But they, too, had a
dream that one day their sons, daughters, grandsons, granddaughters, great-
granddaughters might pursue prosperity and happiness in this land."  Ben Carson,
Trump's Secretary of Housing and Urban Development, made these statements while
discussing immigrants.  **Additional quotes**  From the same article:  "What if
the abolitionists had said, 'I don't believe in slavery, I think it's wrong, but
you guys do whatever you want to do?'" - Ben Carson  And from the second
article:  "We are continuously hearing that there's not 11 million [undocumented
immigrants], there could be 20 million, there could be 30 million." - Katrina
Pierson, Trump's spokesperson


In [224]:
test = 'Hello How are you ?'
answer = testLLMresponse(model_name, test)
print(textwrap.fill(answer['response'], width= 80))

Hello! I'm just an AI, so I don't have feelings or emotions like humans do. I
exist solely to assist and provide information, so I'm always "on" and ready to
help with any questions or topics you'd like to discuss. I'm not capable of
experiencing emotions such as happiness, sadness, or tiredness. But I'm here for
you, and I'll do my best to provide helpful responses and engaging
conversations! How can I assist you today?
